# Modelos de efeitos fixos H1 (Tese Fernando)

- Quinta tentativa, após reuniões com Professores Nicolella e Sakurai (17 e 18/11/2026);
- Análise passo a passo, partindo de OLS e variáveis passo a passo.

In [1]:
## Diagnóstico do Modelo Inicial e Linha de Base (Etapas 1 a 3)

# **Decisões metodológicas documentadas (grill-me e `PRE_ESPECIFICACAO.md`):**
# 1. **Amostra observada (`obs = TRUE`):** o dataframe completo (`df`, 3.384 linhas) contém 612 observações (18,1%) cujo desfecho `y1a_taxa_conversao__*` foi preenchido na base precária (50% interpolação linear municipal + 50% mediana da microrregião×ano). Nos modelos a seguir, restringimos a estimação principal às **2.772 observações efetivamente observadas (137 municípios)**.
# 2. **Tratamento de `multas_ibama`:** na base precária, 48% das linhas de `x3_n_autos_flora__ibama_espacial` receberam zero administrativo. Conforme o texto do artigo (Seção 2.3), ausências não equivalem a zero real. Por isso, retiramos os autos do modelo principal e os avaliamos: (i) na amostra fiscal elegível (`fisc = TRUE`, 1.140 obs, Seção 2.4) e (ii) por meio dos **embargos do IBAMA geocodificados** na amostra completa (Caminho B2, Seção 2.7).
# 3. **Padronização e escala (`ESCOLHENDO VARIAVEIS H1 PREC.txt`):**
#    - Desfecho em pontos percentuais: `y_agr = 100 * y1a_taxa_conversao__agricultura` e `y_pas = 100 * y1a_taxa_conversao__pastagem`.
#    - Preços reais em log natural centrados na média dos 24 anos (`lps`, `lpm`).
#    - Aptidão GAEZ padronizada pelo desvio-padrão populacional dos 141 municípios (`zs`, `zm`).
#    - Interações principais: `xs = lps * zs` e `xm = lpm * zm`.
#    - Controle climático: `rain = x1c_v2__precip_anomaly_1981_2000 / 100` (em 100 mm).

### 0. AMBIENTE

#### 0.1 Bibliotecas e ambiente

In [ ]:
## Carregar pacotes
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(fixest))
suppressPackageStartupMessages(library(sandwich))
suppressPackageStartupMessages(library(lmtest))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(sf))

In [ ]:
# Exibe todas as colunas das tabelas, sem saída LaTeX.
options(
    width = 150, repr.matrix.max.cols = Inf,
    jupyter.display_mimetypes = setdiff(getOption("jupyter.display_mimetypes"), "text/latex")
)

# Configura a execução do fixest para esta análise.
setFixest_nthreads(1)
setFixest_notes(FALSE)

# Ativa as operações geográficas com s2.
sf_use_s2(TRUE)

In [ ]:
# Indica a pasta local do notebook e da base principal.
dir_modelos <- "C:/Users/ferna/Documents/dev/Tese USP/tese_eco_usp/2_INTI/chapters/chap1/stats/2. MODELOS/grupos_h1/2_base_completa_h1_precaria/MODELOS_H1_PREC"

# Usa a pasta do notebook se a base não estiver no diretório atual. Aponta a base:
if (!file.exists("base_modelos_h1_prec.csv")) setwd(dir_modelos)

# Localiza a pasta 2_INTI a partir do diretório do modelo.
root <- normalizePath(file.path(getwd(), "../../../../../../.."), winslash = "/")

# Confere as condições necessárias para seguir com a análise.
stopifnot(basename(root) == "2_INTI")

# Separa os arquivos exportados pela versão 3.
out_dir <- file.path(root, "audit", "modelos_h1_prec_v3_20261118", "saidas")

# Cria a pasta de saída da versão 3.
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

#### 0.2 Leitura e preparação da base

In [ ]:
# Lê indicadores lógicos do CSV.
tf <- function(x) tolower(as.character(x)) == "true"

# Define o desvio-padrão populacional para padronizar aptidões.
sd_pop <- function(x) sqrt(mean((x - mean(x, na.rm = TRUE))^2, na.rm = TRUE))

# Lê a mesma base municipal usada na versão 2.
raw_v2 <- as_tibble(read.csv("base_modelos_h1_prec.csv", stringsAsFactors = FALSE))

# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(raw_v2) == 3384, !anyDuplicated(raw_v2[c("z0_codigo_ibge", "z0_ano")]))

In [ ]:
# Renomear colunas da base (diminuir nome das colunas para facilitar a manipulação)
df2 <- raw_v2 %>% transmute(
  ibge = factor(z0_codigo_ibge),                                 # Código IBGE (fator)
  ibge_int = as.integer(z0_codigo_ibge),                         # Código IBGE (inteiro)
  ano = as.integer(z0_ano),                                      # Ano (inteiro)
  micro = factor(z0_microrregiao),                               # Microrregião (fator)
  bioma = factor(z0_bioma_predominante),                         # Bioma predominante (fator)
  amz = z0_biome_amazonia_share_i,                               # Prop. área municipal Amazônia (numérico)
  cer = z0_biome_cerrado_share_i,                                # Prop. área municipal Cerrado (numérico)
  pan = z0_biome_pantanal_share_i,                               # Prop. área municipal Pantanal (numérico)
  taxa_agr = y1a_taxa_conversao__agricultura,                    # Taxa conv agricultura (numérico)
  y_agr = 100 * y1a_taxa_conversao__agricultura,                 # Taxa conversão agricultura pp (numérico)
  y_pas = 100 * y1a_taxa_conversao__pastagem,                    # Taxa conversão pastagem pp (numérico)
  area_soja_ha = x2b_area_plantada_ha__soja,                     # Área plantada soja ha (numérico)
  area_milho_ha = x2b_area_plantada_ha__milho,                   # Área plantada milho ha (numérico)
  ps_l1 = x2a_preco_real_usd2024_media_anual_lag1__soja,         # Preço real médio ano soja, lag1 (numérico)
  pm_l1 = x2a_preco_real_usd2024_media_anual_lag1__milho,        # Preço real médio ano milho, lag1 (numérico)
  ps_t = x2a_preco_real_usd2024_media_anual__soja,               # Preço real médio ano soja ano corrente (numérico)
  pm_t = x2a_preco_real_usd2024_media_anual__milho,              # Preço real médio ano milho ano corrente (numérico)
  dap_l1 = x2a_preco_real_usd2024_media_anual_lag1__dap,         # Preço real médio ano fertilizante DAP, lag1 (numérico)
  apt_s = x2b_capacidade_valor__soja__gaez_yxHr0_soy,            # Aptidão produtiva soja GAEZ (numérico)
  apt_m = x2b_capacidade_valor__milho__gaez_yxHr0_mze,           # Aptidão produtiva milho GAEZ (numérico)
  cst_s = x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,       # Custo real ha soja 1ª safra, Conab (numérico)
  cst_m = x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,   # Custo real ha milho 2ª safra, Conab (numérico)
  rain = x1c_v2__precip_anomaly_1981_2000 / 100,                 # Chuva 100 mm (numérico)
  autos_l1 = x3_n_autos_flora_elegiveis_lag1__ibama_espacial,    # Nr. autos flora Ibama lag1 (numérico)
  obs = tf(x4_v2__flag_amostra_conversao_ambos),                 # Indicador amostra conversão (lógico)
  fisc = tf(x4_v2__flag_amostra_fiscal_lag1)                     # Indicador elegibilidade amostra fiscal (lógico)
)

In [ ]:
head(df2, 2)

ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc
<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>
5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE
5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE


In [ ]:
str(df2)

tibble [3,384 × 26] (S3: tbl_df/tbl/data.frame)
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:3384] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:3384] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:3384] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:3384] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:3384] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:3384] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ area_milho_ha: num [1:3384] 500 300 300 150 200 240 105

#### 0.3 Defasagem dos preços e padronização da aptidão

In [ ]:
# anos_v2 tem uma linha por ANO, pois os preços de soja, milho e DAP são comuns aos municípios.
# Cada um dos 24 anos entra uma vez na média dos logaritmos dos preços defasados.
# Essas médias são usadas para centrar os logaritmos dos preços em df2.
# Em ordem cronológica, lead() obtém os preços de soja e milho do ano seguinte.
# Esses preços são incorporados a df2 pelo ano para os testes de placebo.

In [ ]:
# Organiza a série anual de preços e suas defasagens.
anos_v2 <- df2 %>% distinct(ano, ps_l1, pm_l1, ps_t, pm_t, dap_l1) %>% arrange(ano) %>%
  mutate(ps_lead = lead(ps_t), pm_lead = lead(pm_t))

In [ ]:
head(anos_v2, 2)

ano,ps_l1,pm_l1,ps_t,pm_t,dap_l1,ps_lead,pm_lead
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2001,385.9825,161.3827,346.9711,158.8466,268.9632,370.6660,173.0334
2002,346.9711,158.8466,370.6660,173.0334,254.3193,450.0622,179.7087


In [ ]:
str(anos_v2)

tibble [24 × 8] (S3: tbl_df/tbl/data.frame)
 $ ano    : int [1:24] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ ps_l1  : num [1:24] 386 347 371 450 510 ...
 $ pm_l1  : num [1:24] 161 159 173 180 186 ...
 $ ps_t   : num [1:24] 347 371 450 510 441 ...
 $ pm_t   : num [1:24] 159 173 180 186 158 ...
 $ dap_l1 : num [1:24] 269 254 265 293 340 ...
 $ ps_lead: num [1:24] 371 450 510 441 418 ...
 $ pm_lead: num [1:24] 173 180 186 158 190 ...


In [ ]:
# mun_v2 tem uma linha por MUNICÍPIO, pois a aptidão GAEZ de soja e milho é fixa no tempo.
# Cada um dos 141 municípios entra uma vez na média e no desvio-padrão populacional da aptidão de cada cultura. Esses valores servem para padronizar as aptidões em df2 antes de construir as interações com os logaritmos centrados dos preços defasados.

In [ ]:
# Seleciona as aptidões fixas de cada município.
mun_v2 <- df2 %>% distinct(ibge, ibge_int, apt_s, apt_m)

In [ ]:
head(mun_v2, 2)

ibge,ibge_int,apt_s,apt_m
<fct>,<int>,<dbl>,<dbl>
5100102,5100102,3665.637,4604.177
5100201,5100201,2971.610,3796.506


In [ ]:
str(mun_v2)

tibble [141 × 4] (S3: tbl_df/tbl/data.frame)
 $ ibge    : Factor w/ 141 levels "5100102","5100201",..: 1 2 3 4 5 6 7 8 9 10 ...
 $ ibge_int: int [1:141] 5100102 5100201 5100250 5100300 5100359 5100409 5100508 5100607 5100805 5101001 ...
 $ apt_s   : num [1:141] 3666 2972 2560 3226 2851 ...
 $ apt_m   : num [1:141] 4604 3797 2894 3468 3298 ...


#### 0.4 Auditoria I

In [ ]:
stopifnot(nrow(anos_v2) == 24, nrow(mun_v2) == 141)
# Interrompe a execução se as tabelas não tiverem 24 linhas anuais e 141 linhas municipais.
# A checagem confere as dimensões antes de centrar os preços e padronizar as aptidões.

In [ ]:
c_ps <- mean(log(anos_v2$ps_l1))
# Calcula a média dos logaritmos do preço da soja defasado em um ano, com peso igual por ano.
# Em 0.5, subtrai-se c_ps do log do preço defasado para obter lps, usado na interação com a aptidão.

In [ ]:
c_pm <- mean(log(anos_v2$pm_l1))
# Calcula a média dos logaritmos do preço do milho defasado em um ano, com peso igual por ano.
# Em 0.5, subtrai-se c_pm do log do preço defasado para obter lpm, usado na interação com a aptidão.

In [ ]:
c_dap <- mean(log(anos_v2$dap_l1))
# Calcula a média dos logaritmos do preço do DAP defasado em um ano, com peso igual por ano.
# Em 0.5, c_dap centra o log do preço defasado antes das interações com as aptidões de soja e milho.

#### 0.5 Base ajustada

In [ ]:
df2 <- df2 %>%
  left_join(anos_v2 %>% select(ano, ps_lead, pm_lead), by = "ano") %>%
  arrange(ibge, ano) %>%
  group_by(ibge) %>%
  mutate(cst_s_l1 = lag(cst_s), cst_m_l1 = lag(cst_m)) %>%
  ungroup() %>%
  mutate(
    lps = log(ps_l1) - c_ps,
    lpm = log(pm_l1) - c_pm,
    zs = (apt_s - mean(mun_v2$apt_s)) / sd_pop(mun_v2$apt_s),
    zm = (apt_m - mean(mun_v2$apt_m)) / sd_pop(mun_v2$apt_m),
    xs = lps * zs,
    xm = lpm * zm,
    trend = (ano - 2001) / 10,
    zs_tr = zs * trend,
    zm_tr = zm * trend,
    xs_f = (log(ps_lead) - c_ps) * zs,
    xm_f = (log(pm_lead) - c_pm) * zm,
    xs_amz = xs * amz, xs_pan = xs * pan,
    xm_amz = xm * amz, xm_pan = xm * pan,
    x_dap_s = (log(dap_l1) - c_dap) * zs,
    x_dap_m = (log(dap_l1) - c_dap) * zm,
    x_cst_s = (log(cst_s_l1) - mean(log(cst_s_l1), na.rm = TRUE)) * zs,
    x_cst_m = (log(cst_m_l1) - mean(log(cst_m_l1), na.rm = TRUE)) * zm,
    f1 = log1p(autos_l1),
    f1_ps = lps * f1, f1_zs = zs * f1, f1_xs = xs * f1,
    f1_pm = lpm * f1, f1_zm = zm * f1, f1_xm = xm * f1,
    post05 = as.numeric(ano >= 2005),
    post06 = as.numeric(ano >= 2006)
  )
# Anexa a df2, pelo ano, os preços futuros de soja e milho para os testes de placebo.
# Ordena por município e ano para calcular as defasagens dos custos dentro de cada município.
# Centra os logaritmos dos preços e padroniza as aptidões entre municípios para construir xs e xm.
# Combina a aptidão com uma tendência em décadas desde 2001 para admitir trajetórias distintas.
# Interage xs e xm com as frações de Amazônia e Pantanal para examinar diferenças por bioma.
# Combina DAP e custos da Conab com as aptidões para avaliar a sensibilidade às medidas de custo.
# Usa log(1 + autos) para manter os zeros e cria interações para os modelos de fiscalização.
# Marca os anos a partir de 2005 e 2006 para comparar recortes temporais alternativos.

In [ ]:
head(df2, 2)

ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE,370.6660,173.0334,NA,NA,-0.3467312,-0.3716491,1.960358,1.67101,-0.6797172,-0.6210295,0.0,0.0000000,0.000000,-0.7590935,-0.5045499,0,0,0,0,-1.105066,-0.9419591,NA,NA,0,0,0,0,0,0,0,0,0
5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE,450.0622,179.7087,690.76,536.61,-0.4532818,-0.3874885,1.960358,1.67101,-0.8885945,-0.6474972,0.1,0.1960358,0.167101,-0.3786187,-0.4412974,0,0,0,0,-1.214815,-1.0355095,-0.58813,-0.6638905,0,0,0,0,0,0,0,0,0


In [ ]:
str(df2)

tibble [3,384 × 58] (S3: tbl_df/tbl/data.frame)
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:3384] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:3384] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:3384] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:3384] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:3384] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:3384] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ area_milho_ha: num [1:3384] 500 300 300 150 200 240 105

In [ ]:
df2 <- df2 %>% mutate(row_id = row_number()) %>% select(row_id, everything())
# Numera as linhas de df2 e põe row_id na primeira coluna para facilitar a conferência da ordem.

In [ ]:
d_obs  <- df2 %>% filter(obs)
# Seleciona as linhas com obs = TRUE para estimar os modelos com conversão observada.

In [ ]:
str(d_obs)

tibble [2,772 × 59] (S3: tbl_df/tbl/data.frame)
 $ row_id       : int [1:2772] 1 2 3 4 5 6 7 8 9 10 ...
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:2772] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:2772] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:2772] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:2772] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:2772] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:2772] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $

In [ ]:
d_fisc <- df2 %>% filter(fisc)
# Seleciona as linhas com fisc = TRUE para analisar os autos de flora na amostra fiscal elegível.

In [ ]:
str(d_fisc)

tibble [1,140 × 59] (S3: tbl_df/tbl/data.frame)
 $ row_id       : int [1:1140] 9 10 23 24 29 32 33 34 35 36 ...
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 2 2 2 2 2 2 ...
 $ ibge_int     : int [1:1140] 5100102 5100102 5100102 5100102 5100201 5100201 5100201 5100201 5100201 5100201 ...
 $ ano          : int [1:1140] 2009 2010 2023 2024 2005 2008 2009 2010 2011 2012 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 9 9 9 9 9 9 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:1140] 0 0 0 0 0.000238 ...
 $ cer          : num [1:1140] 1 1 1 1 1 ...
 $ pan          : num [1:1140] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:1140] 1.97e-05 1.98e-05 0.00 0.00 8.35e-05 ...
 $ y_agr        : num [1:1140] 0.00197 0.00198 0 0 0.00835 ...
 $ y_pas        : num [1:1140] 1.25 1.33 1.95 2.6 2.07 ...
 $ area_soja_ha : num [1:1140] 0 0 0 0 50000 ...
 $ area_milho_ha: num [1:1140] 80 

#### 0.6 Auditoria II

In [ ]:
stopifnot(nrow(d_obs) == 2772, n_distinct(d_obs$ibge) == 137,
          nrow(d_fisc) == 1140, n_distinct(d_fisc$ibge) == 128)
# Confere 2.772 linhas e 137 municípios em d_obs e 1.140 linhas e 128 municípios em d_fisc.
# Interrompe a execução se as amostras diferirem dessas dimensões antes da estimação.

In [ ]:
bioma_amz <- levels(df2$bioma)[grepl("^Amaz", levels(df2$bioma))]
# Recupera o rótulo da Amazônia entre os níveis de bioma pelo prefixo Amaz.
# Esse rótulo é usado para filtrar os municípios de bioma predominante amazônico.

In [ ]:
bioma_cer <- levels(df2$bioma)[grepl("^Cerr", levels(df2$bioma))]
# Recupera o rótulo do Cerrado entre os níveis de bioma pelo prefixo Cerr.
# Esse rótulo é usado para filtrar os municípios de bioma predominante cerrado.

In [ ]:
bioma_pan <- levels(df2$bioma)[grepl("^Pant", levels(df2$bioma))]
# Recupera pelo prefixo Pant o rótulo do Pantanal para permitir um recorte por bioma predominante.
# Os recortes por rótulo nas células seguintes usam a Amazônia e o Cerrado.

#### 0.7 Funções de estimação

In [ ]:
RES <- list()
# Cria a lista que recebe os resultados das estimações e dos testes ao longo do notebook.
# Ao final, os registros são reunidos em uma tabela para consulta e exportação.

In [ ]:
rec <- function(m, id, grupo, descr, amostra = NA_character_) {
  ct <- as.data.frame(coeftable(m))
  r2_t <- tryCatch(as.numeric(r2(m, "r2")), error = function(e) NA_real_)
  r2_w <- tryCatch(as.numeric(r2(m, "wr2")), error = function(e) NA_real_)
  RES[[length(RES) + 1]] <<- tibble(
    id = id, grupo = grupo, descricao = descr, amostra = amostra,
    termo = rownames(ct), estimativa = ct[[1]], erro_padrao = ct[[2]], p = ct[[4]], n = nobs(m),
    r2_total = r2_t, r2_within = r2_w
  )
  invisible(m)
}
# Acrescenta a RES uma linha por termo estimado, com coeficiente, erro-padrão e valor-p.
# Registra o modelo, a amostra, o número de observações e os R² total e within.
# Se o cálculo de um R^2 falhar, registra NA para que os demais resultados sejam guardados.
# Devolve o modelo sem imprimi-lo, para que possa ser atribuído a um objeto e usado nas tabelas.

In [ ]:
fit_run <- function(crop, data, y = "y_agr", fe = "ibge + bioma^ano", vcov = ~ibge,
                    trend = TRUE, rain = TRUE, extra = NULL) {
  x   <- if (crop == "soja") "xs" else "xm"
  ztr <- if (crop == "soja") "zs_tr" else "zm_tr"
  rhs <- c(x, if (trend) ztr, if (rain) "rain", extra)
  f <- as.formula(paste(y, "~", paste(rhs, collapse = " + "), "|", fe))
  feols(f, data = data, vcov = vcov)
}
# Monta uma fórmula comum às duas culturas, escolhendo xs para soja e xm para milho.
# Por padrão, inclui a tendência por aptidão e o controle de chuva.
# Estima com feols, com efeitos fixos de município e de bioma por ano por padrão.
# Usa erros-padrão agrupados por município por padrão. Os argumentos permitem variar o modelo.

In [ ]:
run <- function(crop, data, id, grupo, descr, amostra, ...) {
  rec(fit_run(crop, data, ...), id, grupo, descr, amostra)
}
# Chama fit_run para estimar e rec para registrar o resultado com a identificação e a amostra.
# Isso mantém o mesmo procedimento de registro quando a especificação muda pelos argumentos.

#### 0.8 Formato das tabelas

In [ ]:
# Formata estimativas, erros-padrão e métricas de ajuste para leitura.
tab <- function(..., headers = NULL) {
  ms <- list(...)
  if (is.null(headers)) headers <- paste0("(", seq_along(ms), ")")
  estrelas <- function(p) ifelse(p < 0.01, "***", ifelse(p < 0.05, "**", ifelse(p < 0.10, "*", "")))
  termos <- unique(unlist(lapply(ms, function(m) rownames(coeftable(m)))))
  col <- function(m) {
    ct <- coeftable(m)
    vapply(termos, function(t) {
      if (t %in% rownames(ct)) {
        sprintf("%s%s (%s)", formatC(ct[t, 1], digits = 3, format = "g", flag = "#"),
                estrelas(ct[t, 4]), formatC(ct[t, 2], digits = 3, format = "g", flag = "#"))
      } else ""
    }, character(1))
  }
  out <- as.data.frame(do.call(cbind, lapply(ms, col)), stringsAsFactors = FALSE)
  colnames(out) <- headers; rownames(out) <- termos
  info <- do.call(cbind, lapply(ms, function(m) c(
    `N` = format(nobs(m), big.mark = ""),
    `R2 total` = sprintf("%.4f", as.numeric(r2(m, "r2"))),
    `R2 within` = sprintf("%.4f", as.numeric(r2(m, "wr2"))),
    `Efeitos fixos` = paste(m$fixef_vars, collapse = " + "))))
  colnames(info) <- headers
  print(rbind(out, as.data.frame(info, stringsAsFactors = FALSE)))
  cat("Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01\n")
}

#### 0.9 Validação do Ambiente

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("Ambiente v2 configurado. Amostra observada:", nrow(d_obs), "obs (", n_distinct(d_obs$ibge), "municípios).\n")

Ambiente v2 configurado. Amostra observada: 2772 obs ( 137 municípios).


In [ ]:
# Exibe todas as colunas da base em HTML.
options(repr.matrix.max.cols = Inf)
IRdisplay::display_html(repr::repr_html(head(df2, 2)))

row_id,ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<int>,<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE,370.6660,173.0334,NA,NA,-0.3467312,-0.3716491,1.960358,1.67101,-0.6797172,-0.6210295,0.0,0.0000000,0.000000,-0.7590935,-0.5045499,0,0,0,0,-1.105066,-0.9419591,NA,NA,0,0,0,0,0,0,0,0,0
2,5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE,450.0622,179.7087,690.76,536.61,-0.4532818,-0.3874885,1.960358,1.67101,-0.8885945,-0.6474972,0.1,0.1960358,0.167101,-0.3786187,-0.4412974,0,0,0,0,-1.214815,-1.0355095,-0.58813,-0.6638905,0,0,0,0,0,0,0,0,0


In [ ]:
tail(df2)

row_id,ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<int>,<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3379,5108956,5108956,2019,Alta Floresta,Amazônia,1,0,0,1.756728e-05,1.756728e-03,2.0317592,1509,0,492.8343,205.4161,452.9066,208.6727,491.3879,2652.098,3027.51,887.4231,673.5731,2.4872583,4,TRUE,TRUE,492.6450,200.4740,957.2221,730.7622,-0.1023504,-0.1303901,-0.7098251,-0.3542104,0.07265085,0.04618554,1.8,-1.277685,-0.6375788,0.07292357,0.05481153,0.07265085,0,0.04618554,0,-0.0276496,-0.01379745,-0.01861968,0.03134137,1.6094379,-0.1647265,-1.1424194,0.1169270,-0.2098548,-0.5700797,0.07433276,1,1
3380,5108956,5108956,2020,Alta Floresta,Amazônia,1,0,0,0.000000e+00,0.000000e+00,0.9765656,3800,0,452.9066,208.6727,492.6450,200.4740,376.2261,2652.098,3027.51,872.6322,682.0184,0.4406802,8,TRUE,TRUE,675.8348,300.5185,887.4231,673.5731,-0.1868375,-0.1146605,-0.7098251,-0.3542104,0.13262192,0.04061394,1.9,-1.348668,-0.6729998,-0.15149470,-0.08858149,0.13262192,0,0.04061394,0,0.1619045,0.08079211,0.03512365,0.06020656,2.1972246,-0.4105239,-1.5596451,0.2914002,-0.2519348,-0.7782799,0.08923794,1,1
3381,5108956,5108956,2021,Alta Floresta,Amazônia,1,0,0,0.000000e+00,0.000000e+00,1.1843772,7692,1835,492.6450,200.4740,675.8348,300.5185,378.4231,2652.098,3027.51,771.6819,716.8644,2.7188127,12,TRUE,TRUE,724.1735,341.6679,872.6322,682.0184,-0.1027346,-0.1547429,-0.7098251,-0.3542104,0.07292357,0.05481153,2.0,-1.419650,-0.7084209,-0.20053100,-0.13403735,0.07292357,0,0.05481153,0,0.1577716,0.07872973,0.04705419,0.05579307,2.5649494,-0.2635090,-1.8206653,0.1870453,-0.3969076,-0.9085318,0.14058881,1,1
3382,5108956,5108956,2022,Alta Floresta,Amazônia,1,0,0,1.387445e-05,1.387445e-03,1.9896267,15394,6157,675.8348,300.5185,724.1735,341.6679,694.0289,2652.098,3027.51,811.3913,1220.7631,-1.7536075,3,TRUE,TRUE,615.8341,260.4299,771.6819,716.8644,0.2134254,0.2500815,-0.7098251,-0.3542104,-0.15149470,-0.08858149,2.1,-1.490633,-0.7438419,-0.08550149,-0.03786724,-0.15149470,0,-0.08858149,0,-0.2727379,-0.13609917,0.13432132,0.03814272,1.3862944,0.2958704,-0.9840265,-0.2100163,0.3466866,-0.4910399,-0.12280002,1,1
3383,5108956,5108956,2023,Alta Floresta,Amazônia,1,0,0,3.139054e-07,3.139054e-05,1.1337298,28548,12000,724.1735,341.6679,615.8341,260.4299,828.5465,2652.098,3027.51,1487.0857,1003.5802,-2.8253336,1,TRUE,TRUE,462.7132,190.6186,811.3913,1220.7631,0.2825076,0.3784116,-0.7098251,-0.3542104,-0.20053100,-0.13403735,2.2,-1.561615,-0.7792630,0.11741629,0.07266726,-0.20053100,0,-0.13403735,0,-0.3984900,-0.19885085,0.09870377,-0.15041934,0.6931472,0.1958194,-0.4920132,-0.1389975,0.2622950,-0.2455200,-0.09290761,1,1
3384,5108956,5108956,2024,Alta Floresta,Amazônia,1,0,0,6.342276e-07,6.342276e-05,0.3692473,33773,12847,615.8341,260.4299,462.7132,190.6186,566.6898,2652.098,3027.51,1252.6020,873.4150,-4.2843223,4,TRUE,TRUE,NA,NA,1487.0857,1003.5802,0.1204543,0.1069061,-0.7098251,-0.3542104,-0.08550149,-0.03786724,2.3,-1.632598,-0.8146840,NA,NA,-0.08550149,0,-0.03786724,0,-0.1288553,-0.06430019,-0.33132473,-0.08102869,1.6094379,0.1938637,-1.1424194,-0.1376093,0.1720587,-0.5700797,-0.06094498,1,1


#### 0.10 AED

In [ ]:
ls()

[1] "anos_v2"       "baseline_step" "bioma_amz"     "bioma_cer"     "bioma_pan"     "c_dap"         "c_pm"          "c_ps"          "d_fisc"       
[10] "d_obs"         "df2"           "dir_modelos"   "fit_run"       "mun_v2"        "nb_base"       "out_dir"       "raw_v2"        "rec"          
[19] "RES"           "root"          "run"           "sd_pop"        "tab"           "tf"

In [ ]:
# * nb_base: 1. MODELOS. O primeiro passo usa a base inteira. Os passos 2 a 5 usam filter(nb_base, obs).

# * d_obs: Modelos principais M1, M2 e M3 e a maior parte dos exercícios seguintes. Contém as linhas com obs = TRUE.

# * d_fisc: Modelos da amostra fiscal em 2.4. Contém as linhas com fisc = TRUE.

# * df2: Teste de robustez com o painel completo em 2.4, que inclui as linhas com desfecho preenchido. É também a base de origem de nb_base, d_obs e d_fisc.

In [ ]:
# d_obs e d_fisc são recortes de linhas de df2. d_obs mantém as linhas com obs = TRUE, e d_fisc mantém as linhas com fisc = TRUE. São dois filtros aplicados separadamente a df2.

# nb_base é uma versão ampliada de df2: mantém as 3.384 linhas e acrescenta colunas para a especificação inicial. Na seção 1. MODELOS, o primeiro passo usa nb_base inteira e os passos 2 a 5 aplicam filter(nb_base, obs), selecionando as mesmas linhas de d_obs, mas com as colunas adicionais de nb_base.

***

### 1. MODELOS